# Deep Learning 041 — The Visual Cortex

Hubel and Wiesel found cells that respond to **one orientation of edge and ignore
the rest** (simple cells), feeding cells that respond to that feature **anywhere
in a region** (complex cells). Those two findings are the convolutional layer and
the pooling layer.

This notebook builds both and measures the selectivity.

| Part | What we check |
|---|---|
| A | one filter, several orientations: the response collapses off-angle |
| B | a simple cell is position-specific |
| C | a complex cell (pooling) keeps the answer when the bar moves |

In [ ]:
import numpy as np
np.set_printoptions(precision=1, suppress=True)

def convolve(img, k):
    kh, kw = k.shape
    out = np.zeros((img.shape[0] - kh + 1, img.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i + kh, j:j + kw] * k)
    return out

## Part A — One cell, one orientation

In [ ]:
def bar(angle, n=9):
    """An image containing a single bright bar at 0, 45, 90 or 135 degrees."""
    img = np.zeros((n, n))
    mid = n // 2
    if angle == 0:        img[mid, :] = 1                      # horizontal
    elif angle == 90:     img[:, mid] = 1                      # vertical
    elif angle == 45:     np.fill_diagonal(np.fliplr(img), 1)
    elif angle == 135:    np.fill_diagonal(img, 1)
    return img

horizontal_cell = np.array([[-1, -1, -1],
                            [ 2,  2,  2],
                            [-1, -1, -1]])

print("a cell tuned to horizontal edges, shown bars at four angles:")
responses = {}
for angle in (0, 45, 90, 135):
    r = convolve(bar(angle), horizontal_cell).max()
    responses[angle] = r
    print(f"  bar at {angle:>3}deg -> peak response {r:>5.1f}")

assert responses[0] == max(responses.values())
print("\nStrongest at its own orientation, far weaker at the others - which is")
print("what Hubel and Wiesel recorded from a single cell in V1.")

## Part B — Simple cell: the right feature, but only here

Move the bar and the same cell's *location* of response moves with it.

In [ ]:
img_a = np.zeros((9, 9)); img_a[2, :] = 1      # bar near the top
img_b = np.zeros((9, 9)); img_b[6, :] = 1      # same bar, lower down

fa, fb = convolve(img_a, horizontal_cell), convolve(img_b, horizontal_cell)
print("peak position with the bar at row 2:", np.unravel_index(fa.argmax(), fa.shape))
print("peak position with the bar at row 6:", np.unravel_index(fb.argmax(), fb.shape))
print("feature maps identical?", np.array_equal(fa, fb))

## Part C — Complex cell: the same feature, anywhere

Max pooling asks "did this feature appear in this window?" and throws away where
exactly. That is the complex cell.

In [ ]:
def max_pool(x, k=7):
    """One big window over the whole map: 'did it fire anywhere?'"""
    return x.max()

print("simple cell (conv) output differs when the bar moves:", not np.array_equal(fa, fb))
print("complex cell (pooled) answer for bar at row 2:", max_pool(fa))
print("complex cell (pooled) answer for bar at row 6:", max_pool(fb))
assert max_pool(fa) == max_pool(fb)
print("\nSame answer: the feature is present. That is translation tolerance,")
print("and it is why conv -> pool is the pairing in every CNN.")

## What to take away

- A **simple cell** responds to one orientation **at one place** → convolution.
- A **complex cell** responds to that feature **anywhere in a region** → pooling.
- This analogy is **specific and documented**, unlike the loose neuron comparison
  lesson 002 warned about.

## Exercises

1. Build a vertical-edge cell and show it is near-silent on a horizontal bar.
2. Pool over 3×3 windows instead of the whole map. How far can the bar move
   before the answer changes?
3. Stack two cells (horizontal and vertical) and record both responses. What
   does a corner look like?